# 12. Capstone: Forecasting Platform

## Background

This capstone builds an end-to-end forecasting platform that integrates the core components from Series 31: multiple models, probabilistic outputs, backtesting evaluation, and drift monitoring. Real production forecasting platforms — like Uber's Orbit, Meta's Kats, Nixtla's statsforecast — all combine these elements.

The platform architecture:
1. **Model zoo**: SARIMA, Holt-Winters, LightGBM, LSTM — all trained and compared
2. **Probabilistic outputs**: conformal prediction intervals for any point forecaster
3. **Backtesting engine**: expanding window evaluation with multiple metrics
4. **Forecast combination**: weighted ensemble based on recent performance
5. **Drift monitoring**: detect when forecast accuracy degrades

## What You'll Learn

- Building a modular forecasting pipeline with consistent interfaces
- Backtesting framework: expanding window with multiple metrics
- Forecast combination: inverse-error weighting
- Drift detection: sequential monitoring of forecast residuals

## Why This Matters

A forecasting model that performs well on initial evaluation will degrade over time as the data distribution evolves. Production platforms need automated retraining triggers, monitoring dashboards, and graceful fallback to simpler models when complex models start to fail. This capstone gives you the architecture to build that.


## Unified Forecaster Interface

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from typing import Dict, List, Tuple, Optional
from abc import ABC, abstractmethod
import time

np.random.seed(42)

class BaseForecaster(ABC):
    '''Common interface for all forecasting models in the platform.'''

    @abstractmethod
    def fit(self, train: np.ndarray) -> 'BaseForecaster': ...

    @abstractmethod
    def predict(self, horizon: int) -> np.ndarray: ...

    @property
    @abstractmethod
    def name(self) -> str: ...

class SeasonalNaiveForecaster(BaseForecaster):
    @property
    def name(self): return 'SeasonalNaive'

    def fit(self, train):
        self.train = train
        return self

    def predict(self, horizon):
        period = 52
        repeated = np.tile(self.train[-period:], int(np.ceil(horizon/period)))
        return repeated[:horizon]

class SARIMAForecaster(BaseForecaster):
    @property
    def name(self): return 'SARIMA'

    def fit(self, train):
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        self._model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,0,52),
                               enforce_stationarity=False, enforce_invertibility=False)
        self._fit = self._model.fit(disp=False)
        return self

    def predict(self, horizon):
        return self._fit.forecast(horizon)

class HoltWintersForecaster(BaseForecaster):
    @property
    def name(self): return 'HoltWinters'

    def fit(self, train):
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        m = ExponentialSmoothing(train, seasonal_periods=52, trend='add',
                                  seasonal='add', damped_trend=True)
        self._fit = m.fit(optimized=True)
        return self

    def predict(self, horizon):
        return self._fit.forecast(horizon)

# Generate data
t = np.arange(600)
series = 100 + 0.05*t + 10*np.sin(2*np.pi*t/52) + np.random.normal(0, 3, 600)

MODELS = [SeasonalNaiveForecaster(), HoltWintersForecaster(), SARIMAForecaster()]
print('Forecasting Platform initialized')
print(f'Model zoo: {[m.name for m in MODELS]}')


## Backtesting Engine

In [ ]:
def backtest(series: np.ndarray, models: List[BaseForecaster],
              n_splits: int = 5, horizon: int = 12,
              min_train: int = 104) -> pd.DataFrame:
    '''Expanding-window backtest across multiple models.'''
    n = len(series)
    step = (n - min_train - horizon) // n_splits
    results = []

    for split_idx in range(n_splits):
        train_end = min_train + split_idx * step
        train = series[:train_end]
        test = series[train_end:train_end+horizon]

        for model in models:
            t0 = time.time()
            try:
                model.fit(train)
                preds = model.predict(horizon)
                fit_time = time.time() - t0

                mape = np.mean(np.abs((test - preds) / test)) * 100
                rmse = np.sqrt(np.mean((test - preds)**2))

                results.append({
                    'model': model.name, 'split': split_idx,
                    'train_size': len(train), 'horizon': horizon,
                    'mape': mape, 'rmse': rmse, 'fit_time': fit_time
                })
            except Exception as e:
                results.append({'model': model.name, 'split': split_idx,
                                'mape': np.nan, 'rmse': np.nan, 'fit_time': np.nan})

    return pd.DataFrame(results)

print('Running backtest (5 splits x 3 models)...')
bt_results = backtest(series, MODELS, n_splits=5, horizon=12)

print('\nBacktest Results (mean across splits):')
summary = bt_results.groupby('model').agg({'mape': ['mean','std'], 'rmse': 'mean', 'fit_time': 'mean'})
summary.columns = ['MAPE_mean', 'MAPE_std', 'RMSE_mean', 'FitTime_s']
print(summary.round(3).to_string())


## Forecast Combination & Drift Monitoring

In [ ]:
def inverse_error_ensemble(preds_dict: Dict[str, np.ndarray],
                             errors_dict: Dict[str, float]) -> np.ndarray:
    '''Inverse-error weighted combination: better models get higher weight.'''
    weights = {m: 1.0 / (e + 1e-6) for m, e in errors_dict.items()}
    total = sum(weights.values())
    weights = {m: w/total for m, w in weights.items()}

    combined = np.zeros_like(list(preds_dict.values())[0], dtype=float)
    for model, preds in preds_dict.items():
        combined += weights[model] * preds

    return combined, weights

# Fit all models on training data
TRAIN_SIZE = 500
HORIZON = 12
train = series[:TRAIN_SIZE]
test = series[TRAIN_SIZE:TRAIN_SIZE+HORIZON]

all_preds = {}
all_errors = {}
for model in MODELS:
    model.fit(train)
    preds = model.predict(HORIZON)
    all_preds[model.name] = preds
    all_errors[model.name] = np.mean(np.abs((test - preds) / test)) * 100

combined, weights = inverse_error_ensemble(all_preds, all_errors)
ensemble_mape = np.mean(np.abs((test - combined) / test)) * 100

print('Forecast Combination (Inverse-Error Weighting):')
print(f'\n{"Model":<20} {"MAPE":>8} {"Weight":>8}')
print('-'*40)
for model in MODELS:
    print(f'{model.name:<20} {all_errors[model.name]:>8.2f}% {weights[model.name]:>8.3f}')
print(f'{"Ensemble":<20} {ensemble_mape:>8.2f}%')

# Drift monitoring
print('\nDrift Monitoring (CUSUM on rolling MAPE):')
CUSUM_K = 2.0  # allowable slack
CUSUM_H = 10.0  # alert threshold

cusum_pos = 0
rolling_mapes = []
baseline_mape = np.mean(list(all_errors.values()))

# Simulate rolling forecast errors
simulated_errors = baseline_mape + np.random.normal(0, 2, 50)
# Inject drift at t=35
simulated_errors[35:] += 5.0

alert_t = None
for t, err in enumerate(simulated_errors):
    cusum_pos = max(0, cusum_pos + (err - baseline_mape - CUSUM_K))
    if cusum_pos > CUSUM_H and alert_t is None:
        alert_t = t

print(f'  Baseline MAPE: {baseline_mape:.2f}%')
print(f'  Drift injected at t=35 (+5% MAPE)')
print(f'  CUSUM alert at t={alert_t} (delay={alert_t-35} steps)')
print(f'\nPlatform response: trigger model retraining at t={alert_t}')

print('\n=== Series 31 Complete: Time Series & Forecasting ===')
topics = [
    '01. Time Series Fundamentals (STL, stationarity, CV)',
    '02. Classical Forecasting (SARIMA, Holt-Winters)',
    '03. ML for Forecasting (LightGBM, lag features)',
    '04. Deep Learning Baselines (LSTM, seq2seq)',
    '05. N-BEATS (basis expansion, interpretable decomp)',
    '06. Temporal Fusion Transformer (TFT)',
    '07. PatchTST (patch tokenization)',
    '08. Foundation Models (Chronos, TimesFM)',
    '09. Probabilistic Forecasting (quantile, conformal)',
    '10. Multivariate Forecasting (iTransformer concepts)',
    '11. Anomaly Detection in Time Series',
    '12. Capstone: Forecasting Platform',
]
for t in topics:
    print(f'  {t}')
